# Tool-Router Ladder: Stage B, LoRA SFT of Qwen2.5-3B (V1-17)

Same data, same hyperparameters as the 0.5B and 1.5B adapters; only the base model changes.

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, the tool-router dataset attached.
**Secrets:** `HF_TOKEN` (write) and `WANDB_API_KEY`, both ticked.
**Run:** Save Version → **Save & Run All (Commit)**. Checkpoints go to the Hub every epoch and a
rerun resumes from them. **No API quota**: this can run on a day the Mac harvest is running.

**Do not install vLLM in this notebook.** It replaces torch. Generation has its own notebook.

**Send back:** the memory-check lines and the last `[train] done: ...` line.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "WANDB_API_KEY"])

In [ ]:
# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

!pip install -q peft
# Kaggle ships torchao 0.10.0; peft's LoRA dispatcher raises on anything below
# 0.16.0 instead of skipping it. Nothing here uses torchao, and with the package
# gone peft's is_torchao_available() returns False and dispatch falls through.
!pip uninstall -y -q torchao

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

### 1. Memory check: one forward and backward on the worst-case 12,176-token episode

Measured on a T4 in September: 5.43 GiB at 0.5B and 8.38 GiB at 1.5B, of 14.56. 3B's weights
alone are 6.2 GB, so it may not fit one card. This tries one card first; if that runs out of
memory, it splits the layers across both T4s (`--shard`: the same arithmetic, one card computing
at a time) and the training cell uses whichever worked.

In [ ]:
import gc, torch
from pathlib import Path
from src.config import load_config
from src.train import load_base_model, masked_lm_loss, wrap_lora

L, N_TARGET = 12176, 1300   # longest episode; 137,149 target tokens / 108 episodes
# GiB reserved per card. The T4 has 14.56, but this kernel keeps ~1.2 after the check, the
# training process needs its own CUDA context, and the optimizer state is not measured here.
# The first 3B run passed a 13.5 limit and then ran out of memory at ~13.4 GiB on one card.
LIMIT = 12.0

def check(name, shard):
    gc.collect(); torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        torch.zeros(1, device=f"cuda:{i}")  # reset fails on a card CUDA has not touched yet
        torch.cuda.reset_peak_memory_stats(i)
    cfg = load_config(Path(f"configs/{name}.yaml"))
    dev = torch.device("cuda")
    model = wrap_lora(cfg, load_base_model(cfg, dev, shard=shard)); model.train()
    ids = torch.randint(0, 150000, (1, L), device=dev)
    labels = torch.full((1, L), -100, device=dev)
    labels[:, -N_TARGET:] = ids[:, -N_TARGET:]
    with torch.autocast("cuda", dtype=torch.float16):
        loss_sum, n = masked_lm_loss(model, ids, labels)
    (loss_sum / n).backward()
    peaks = [torch.cuda.max_memory_reserved(i) / 2**30 for i in range(torch.cuda.device_count())]
    print(f"  {cfg.model.base_model} shard={shard}: peak per card " + ", ".join(f"{p:.2f}" for p in peaks) + " GiB")
    del model; gc.collect(); torch.cuda.empty_cache()
    return max(peaks)

print(f"one episode at {L:,} tokens; {torch.cuda.device_count()} GPU(s)")
try:
    SHARD = check("qwen3b", shard=False) > LIMIT
except torch.OutOfMemoryError:
    print("  one card: out of memory")
    SHARD = True
if SHARD:
    assert check("qwen3b", shard=True) <= LIMIT, "3B does not fit even split across both T4s: stop and send this output"
print("SHARD =", SHARD)

### 2. Qwen2.5-3B: about 21 optimizer steps (1.5B took 51 minutes; expect roughly twice that)

In [ ]:
FLAG = "--shard" if SHARD else ""
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train.py --config configs/qwen3b.yaml --data "$DATA" --output-dir /kaggle/working/qwen3b --hub-repo "$HF_USER/tool-router-qwen3b-sft" --wandb {FLAG}